1 Setup libraries

In [26]:
import json
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

In [27]:
# Set display options for better visibility in the notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Define file paths (this is my path, change to your)
input_file = r'C:\Users\Admin\Documents\ML-cos3049\ML\data\data.json'
output_file = r'C:\Users\Admin\Documents\ML-cos3049\ML\data\data_labeled.json'

2 Get the file path

In [28]:
# Read data
df = pd.read_json(input_file) 
display(df.head()) # Show the first few rows in the notebook


,Transaction ID,Sender Account ID,Receiver Account ID,Transaction amount,Timestamp,Transaction Detail,Geological,Device Use,Date of Birth,Gender,Location,Account balance,Transaction Count,Working Status,Salary (per month)
0,TXN_300001,ACC_04483,REC_7570,458000,2025-02-27 21:37:00,Starbucks,HCMC - VN,Android Phone,1996-08-12,Male,Hai Phong,"79,000,000",3,Employed,"13,000,000"
1,TXN_300002,ACC_04770,REC_8777,203000,2025-04-21 22:53:00,Starbucks,Can Tho - VN,Samsung S23,2004-06-17,Female,Bac Ninh,"29,000,000",2,Employed,"43,000,000"
2,TXN_300003,ACC_06821,REC_3980,1392000,2025-12-17 07:15:00,Restaurant,Hanoi - VN,Samsung S23,1992-05-07,Female,Hue,"140,000,000",4,Employed,"45,000,000"
3,TXN_300004,ACC_08825,REC_7662,11046000,2025-06-25 07:22:00,Monthly Salary,HCMC - VN,iPhone 15,1984-04-25,Female,Can Tho,"21,000,000",2,Employed,"29,000,000"
4,TXN_300005,ACC_03949,REC_6068,120000,2025-12-16 10:09:00,Restaurant,HCMC - VN,Web Browser,1976-10-05,Female,Bac Ninh,"468,000,000",4,Unemployed,0


3 Processing data


In [ ]:
# Processing Data
# 1. Clean the number columns
numeric_columns = ['Transaction amount', 'Account balance', 'Salary (per month)']
for column in numeric_columns:
    df[column] = df[column].astype(str)
    df[column] = df[column].str.replace(',', '', regex=False).str.replace('.', '', regex=False)
    df[column] = pd.to_numeric(df[column], errors='coerce').fillna(0)

# 2. Time processing
if 'Timestamp' in df.columns:
    df['DateTime'] = pd.to_datetime(df['Timestamp'], unit='ms')
    df['Hour'] = df['DateTime'].dt.hour
    df['DayOfWeek'] = df['DateTime'].dt.dayofweek

# 3. Label Encoding
text_columns = ['Transaction Detail', 'Geological', 'Device Use', 'Gender', 'Location', 'Working Status']
df_processed = df.copy()

for column in text_columns:
    if column in df.columns:
        encoder = LabelEncoder()
        new_column_name = column + "_encoded"
        df_processed[new_column_name] = encoder.fit_transform(df[column].astype(str))

df_processed.head()

4 Picking features and Training model

In [ ]:
features = [
    'Transaction amount', 'Account balance', 'Salary (per month)',
    'Hour', 'DayOfWeek',
    'Transaction Detail_encoded', 'Geological_encoded', 
    'Device Use_encoded', 'Location_encoded', 'Working Status_encoded'
]

selected_features = [col for col in features if col in df_processed.columns]
X = df_processed[selected_features]

model = IsolationForest(n_estimators=100, contamination=0.15, random_state=42)
model.fit(X)

In [ ]:
# Predict (-1 is Anomaly, 1 is Normal)
predictions = model.predict(X)

# Assign labels: 1 = Fraud, 0 = Normal
df['is_fraud'] = [1 if p == -1 else 0 for p in predictions]
df['anomaly_score'] = model.score_samples(X)

fraud_count = df['is_fraud'].sum()
print(f"Found {fraud_count} suspicious transactions.")

Show top 5 anomalies

In [ ]:
# Top 5 anomalies
# Filter Fraud transactions and sort by anomaly score
frauds = df[df['is_fraud'] == 1].sort_values('anomaly_score')

columns_to_show = [
    'Transaction ID', 'Transaction amount', 'Transaction Detail', 
    'Location', 'anomaly_score'
]

frauds[columns_to_show].head(5)